# Преамбула и дисклеймер

Этот файл собран на основе исходного `Задание.txt`. Официальные Markdown-формулировки задания сохранены без редактирования, включая спорные формулировки и опечатки. Заполняются только предусмотренные кодовые ячейки; пояснения добавляются отдельными блоками.

## Как уточняется постановка задачи

1. Единственным источником рабочей реализации считается `scraper.py`; ноутбук показывает её применение и результаты проверки.
2. Данные сохраняются как JSON в UTF-8, хотя расширение файла по заданию остаётся `.txt`.
3. Пагинация идёт по фактической ссылке `next`, а не по жёстко заданному числу страниц.
4. Сетевые запросы имеют тайм-аут и проверку HTTP-статуса.
5. Автотесты не зависят от живого сайта и используют локальные HTML-фикстуры.
6. Время `19:00` трактуется как локальное время компьютера.
7. Бесконечный цикл планировщика не запускается при `Run All`.
8. GitHub-публикация не выполняется без отдельного явного запроса.

Эти гипотезы не подменяют требования преподавателя: они делают неоднозначные места проверяемыми и воспроизводимыми.

# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



## Комментарии по доработкам и гипотезам — окружение

- **Исправление:** установка пакетов из ячейки заменена проверкой зависимостей; полный список хранится в `requirements.txt`.
- **Гипотеза:** блокнот запускается из корня проекта либо из папки `notebooks`.
- **Зачем:** повторный запуск не должен менять глобальное окружение или зависеть от текущего каталога.
- **Влияние:** перед первым запуском окружение устанавливается один раз по инструкции из `README.md`.

In [1]:
# Исходная команда шаблона была предназначена для ручной установки:
# ! pip install -q schedule pytest  # установка библиотек, если ещё не
#
# В проекте зависимости устанавливаются один раз из requirements.txt.
# Эта ячейка только проверяет, что нужные пакеты доступны в текущем ядре.
from importlib.util import find_spec

required_packages = {
    "bs4": "beautifulsoup4",
    "pytest": "pytest",
    "requests": "requests",
    "schedule": "schedule",
}
missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if find_spec(module_name) is None
]

if missing_packages:
    raise RuntimeError(
        "Не установлены зависимости: " + ", ".join(missing_packages)
    )

print("Зависимости доступны; установка из ноутбука не требуется.")

Зависимости доступны; установка из ноутбука не требуется.


In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup

from pathlib import Path
import subprocess
import sys


PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "scraper.py").exists():
    raise RuntimeError(f"Не найден scraper.py в {PROJECT_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scraper import (
    DEFAULT_OUTPUT_PATH,
    SCHEDULE_TAG,
    configure_schedule,
    get_book_data as project_get_book_data,
    load_books,
    parse_book_html,
    scrape_books as project_scrape_books,
)

print(f"Корень проекта: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")

Корень проекта: C:\Users\User\Desktop\ДЗ_МФТИ\Python_1211\books_scraper
Python: 3.12.0


## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

### Комментарии по доработкам и гипотезам — задание 1

- **Исправление:** сетевой запрос отделён от функции `parse_book_html`, разбирающей уже полученный HTML.
- **Гипотеза:** цена хранится как `float`, рейтинг как `int`, количество как `int | None`, описание как `str | None`.
- **Зачем:** разбор страницы можно проверить без интернета на фиксированном HTML.
- **Влияние:** требуемая `get_book_data` сохранена, но её результат имеет явный контракт и типы.

In [3]:
def get_book_data(book_url: str) -> dict:
    """Получить нормализованные данные об одной книге.

    Args:
        book_url: Абсолютный URL страницы книги на Books to Scrape.

    Returns:
        Словарь с URL, названием, ценой, рейтингом, наличием,
        количеством, описанием и таблицей Product Information.

    Raises:
        requests.RequestException: Если страницу не удалось загрузить.
        ValueError: Если обязательные элементы страницы отсутствуют.
    """

    return project_get_book_data(book_url)

In [4]:
# Используйте для самопроверки
book_url = (
    "https://books.toscrape.com/catalogue/"
    "a-light-in-the-attic_1000/index.html"
)
RUN_NETWORK_DEMO = False

if RUN_NETWORK_DEMO:
    book_data = get_book_data(book_url)
    source = "живая страница сайта"
else:
    fixture_path = PROJECT_ROOT / "tests" / "fixtures" / "book_page.html"
    fixture_html = fixture_path.read_text(encoding="utf-8")
    book_data = parse_book_html(fixture_html, book_url)
    source = "локальная HTML-фикстура"

required_keys = {
    "url",
    "title",
    "price_gbp",
    "rating",
    "availability",
    "stock_count",
    "description",
    "product_information",
}
assert set(book_data) == required_keys
print(f"Источник проверки: {source}")
book_data

Источник проверки: локальная HTML-фикстура

{'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html',
 'title': 'A Light in the Attic',
 'price_gbp': 51.77,
 'rating': 3,
 'availability': 'In stock (22 available)',
 'stock_count': 22,
 'description': 'Учебное описание книги.',
 'product_information': {'UPC': 'a897fe39b1053632',
  'Product Type': 'Books',
  'Price (excl. tax)': '£51.77',
  'Price (incl. tax)': '£51.77',
  'Tax': '£0.00',
  'Availability': 'In stock (22 available)',
  'Number of reviews': '0'}}

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

### Комментарии по доработкам и гипотезам — задание 2

- **Исправление:** следующая страница определяется по ссылке `next`, абсолютные адреса собираются через `urljoin`.
- **Гипотеза:** `max_pages=None` означает полный каталог; положительное значение используется только для быстрой демонстрации.
- **Зачем:** число страниц сайта может измениться, а цикл пагинации должен быть обнаружен явно.
- **Влияние:** обычный вызов собирает весь каталог; `is_save=True` сохраняет тот же список в `books_data.txt`.

In [5]:
def scrape_books(is_save: bool = False, **kwargs) -> list[dict]:
    """Собрать данные обо всех книгах каталога.

    Args:
        is_save: Сохранить ли результат в ``books_data.txt``.
        **kwargs: Дополнительные параметры проверяемой реализации:
            путь сохранения, тайм-аут, задержка и лимит страниц.

    Returns:
        Список нормализованных словарей с данными книг.

    Raises:
        requests.RequestException: Если сетевой запрос завершился ошибкой.
        RuntimeError: Если обнаружен цикл пагинации.
        ValueError: Если передан некорректный лимит страниц.
    """

    return project_scrape_books(is_save=is_save, **kwargs)

In [6]:
# Проверка работоспособности функции.
# Полный сетевой запуск соответствует исходной строке:
# res = scrape_books(is_save=True)
# Чтобы Run All не выполнял повторно более тысячи запросов, используем
# уже сформированный тем же кодом артефакт. При его отсутствии запускается
# демонстрационный сбор одной страницы.
artifact_path = PROJECT_ROOT / "artifacts" / "books_data.txt"

if artifact_path.exists():
    res = load_books(artifact_path)
    source = "готовый полный артефакт"
else:
    res = scrape_books(is_save=True, max_pages=1)
    source = "сетевая демонстрация одной страницы"

assert isinstance(res, list)
assert all(isinstance(book, dict) for book in res)
print("Источник:", source)
print(type(res), len(res))

Источник: готовый полный артефакт
<class 'list'> 1000


## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



### Комментарии по доработкам и гипотезам — задание 3

- **Исправление:** настройка расписания отделена от бесконечного цикла ожидания.
- **Гипотеза:** `19:00` относится к часовому поясу локальной операционной системы.
- **Зачем:** выполнение всех ячеек должно завершаться, а не зависать в `while True`.
- **Влияние:** ноутбук проверяет созданное задание; постоянный процесс запускается отдельно через `scraper.py --schedule`.

In [7]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
# Создаём задание на 19:00, но не запускаем бесконечный цикл в ноутбуке.
job = configure_schedule("19:00", output_path=DEFAULT_OUTPUT_PATH)

print("Созданное задание:", job)
print("Задач с тегом проекта:", len(schedule.get_jobs(SCHEDULE_TAG)))
print("Постоянный запуск: python scraper.py --schedule --at 19:00")

# Очищаем очередь после демонстрации, чтобы повторный Run All был стабильным.
schedule.clear(SCHEDULE_TAG)
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

Созданное задание: Job(interval=1, unit=days, do=scrape_books, args=(), kwargs={'is_save': True, 'output_path': WindowsPath('C:/Users/User/Desktop/ДЗ_МФТИ/Python_1211/books_scraper/artifacts/books_data.txt'), 'request_delay': 0.05})
Задач с тегом проекта: 1
Постоянный запуск: python scraper.py --schedule --at 19:00


## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


### Комментарии по доработкам и гипотезам — задание 4

- **Исправление:** путь из шаблона `test/test_scraper.py` заменён на фактический `tests/test_scraper.py`.
- **Гипотеза:** модульные тесты должны проверять наш код, а не доступность внешнего сайта.
- **Зачем:** сетевой сбой не должен давать ложное падение домашней работы.
- **Влияние:** тесты используют фикстуры и поддельную HTTP-сессию, но дополнительно был выполнен полный сетевой сбор.

In [8]:
# Ячейка для демонстрации работоспособности.
# В исходном шаблоне указан путь test/test_scraper.py; фактическая папка
# по требованиям задания называется tests, поэтому запускаем её явно.
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_scraper.py"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=False,
)

print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr)
assert test_result.returncode == 0, "Тесты завершились с ошибкой"

...........                                                              [100%]
11 passed in 0.26s



## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

### Комментарии по доработкам и гипотезам — задание 5

- **Исправление:** локальная готовность проекта проверяется до публикации.
- **Гипотеза:** создание репозитория, отправка ветки и Pull Request являются отдельными внешними действиями.
- **Зачем:** проверка структуры не требует доступа к аккаунту GitHub и не создаёт публикаций без разрешения.
- **Влияние:** обязательные локальные файлы готовы; GitHub-этап остаётся неподтверждённым до отдельного запроса.

In [9]:
expected_paths = [
    "scraper.py",
    "README.md",
    "requirements.txt",
    ".gitignore",
    "artifacts/books_data.txt",
    "tests/test_scraper.py",
    "notebooks/HW_03_python_ds_2025.ipynb",
]

missing_paths = []
for relative_path in expected_paths:
    exists = (PROJECT_ROOT / relative_path).exists()
    print(f"{'OK' if exists else 'НЕТ'}: {relative_path}")
    if not exists:
        missing_paths.append(relative_path)

assert not missing_paths, f"Не найдены файлы: {missing_paths}"

OK: scraper.py
OK: README.md
OK: requirements.txt
OK: .gitignore
OK: artifacts/books_data.txt
OK: tests/test_scraper.py
OK: notebooks/HW_03_python_ds_2025.ipynb


## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: ...`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: ...`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: ...`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: ...`

**5. Самооценка по критериям задания**:

`Ответ: ...`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: ...`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

